In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages

In [2]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [3]:
llm = ChatOllama(
    model='llama3:8b-instruct-q4_k_m'
)

In [4]:
result = llm.invoke("What is the capital of India?")
print(result)

content='The capital of India is New Delhi.' additional_kwargs={} response_metadata={'model': 'llama3:8b-instruct-q4_k_m', 'created_at': '2026-03-26T15:03:52.7772204Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7941847300, 'load_duration': 5793828800, 'prompt_eval_count': 17, 'prompt_eval_duration': 667863300, 'eval_count': 9, 'eval_duration': 1466792300, 'logprobs': None, 'model_name': 'llama3:8b-instruct-q4_k_m', 'model_provider': 'ollama'} id='lc_run--019d2aac-3fbf-7b00-9741-d11b620d36a7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 17, 'output_tokens': 9, 'total_tokens': 26}


In [5]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)

    return {'messages': [response]}

In [6]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [7]:
thread_id = '1'
while True:
    user_message = input("Enter the Query: ")
    print('User :', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id' : thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)

    print('AI: ', response['messages'][-1].content)

User : My name is Omkar
AI:  Nice to meet you, Omkar! It's great to have you here. Is there something I can help you with or would you like to chat about something in particular?
User : What is my name
AI:  Your name is Omkar!
User : 
AI:  That's right! I remember: your name is Omkar.
User : exit
